# Course 3 — Cross-Validation and the Bootstrap

How do we honestly estimate how well a model will perform on new data?
Training error lies — this course teaches the two most important tools
for answering that question without a separate test set.

**Sections**
1. Training error vs test error (0:00–0:20)
2. Validation-set approach (0:20–0:40)
3. K-fold cross-validation (0:40–1:15)
4. CV: right and wrong (1:15–1:30)
5. The bootstrap (1:30–2:00)

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split, KFold, LeaveOneOut,
    cross_val_score, GridSearchCV,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import mean_squared_error
import scipy.stats

rng = np.random.default_rng(42)
print('Setup complete.')


## 1. Training error vs test error

### Why training error misleads

The **training error** is the average loss on the data used to fit the model.
The **test error** is the average loss on *new* observations the model has never seen.

More flexible models always reduce training error — but they can overfit the
training noise, causing test error to *rise*. This is the bias-variance trade-off
appearing as a U-shaped test error curve.

In [ ]:
# Generate data from a known quadratic truth: y = 1 + 2x - x^2 + noise
n_train, n_test = 80, 500
x_train = rng.uniform(-3, 3, n_train)
x_test  = rng.uniform(-3, 3, n_test)
noise_sd = 2.0
f_true   = lambda x: 1 + 2*x - x**2

y_train = f_true(x_train) + rng.normal(0, noise_sd, n_train)
y_test  = f_true(x_test)  + rng.normal(0, noise_sd, n_test)

# Fit polynomials of degree 1, 2, 3, 5, 9, 12
degrees = [1, 2, 3, 5, 9, 12]
mse_train, mse_test = [], []

X_tr = x_train.reshape(-1, 1)
X_te = x_test.reshape(-1, 1)

for d in degrees:
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=d, include_bias=False)),
        ('lr',   LinearRegression()),
    ])
    pipe.fit(X_tr, y_train)
    mse_train.append(mean_squared_error(y_train, pipe.predict(X_tr)))
    mse_test.append( mean_squared_error(y_test,  pipe.predict(X_te)))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(degrees, mse_train, 'o-', color='steelblue', label='Training MSE')
ax.plot(degrees, mse_test,  'o-', color='tomato',    label='Test MSE')
ax.axhline(noise_sd**2, color='grey', linestyle='--', label=f'Irreducible error (σ²={noise_sd**2})')
ax.set_xlabel('Polynomial degree (model complexity)')
ax.set_ylabel('MSE')
ax.set_title('Training error deceives — test error is U-shaped')
ax.legend()
plt.tight_layout()
plt.show()

print('Degree | Train MSE | Test MSE')
for d, tr, te in zip(degrees, mse_train, mse_test):
    print(f'  {d:2d}   |  {tr:7.3f}  | {te:7.3f}')


Training MSE falls as degree increases and goes to nearly zero for degree 12.
Test MSE forms a U-shape with a minimum at degree 2 — matching the true data-generating
function. Higher-degree models memorise noise rather than learning the signal.

The irreducible error floor (σ² = 4) is the minimum possible test MSE — even a perfect
model cannot beat it because observations have noise.

## 2. Validation-set approach

### What it is

Split the available data randomly into a **training set** and a **validation (hold-out) set**.
Fit on the training set, evaluate on the validation set.
The validation MSE estimates the test error.

We use the **Auto** dataset (392 cars) to predict `mpg` from `horsepower` using
polynomial regression of degree 1–10.

In [ ]:
# Load Auto data (mpg vs horsepower)
auto = sns.load_dataset('mpg').dropna(subset=['mpg', 'horsepower'])
X_auto = auto['horsepower'].to_numpy().reshape(-1, 1)
y_auto = auto['mpg'].to_numpy()
print(f'Auto dataset: {len(auto)} observations')
print(f'horsepower range: [{X_auto.min():.0f}, {X_auto.max():.0f}]')


In [ ]:
# Single 50/50 validation-set split
X_tr, X_val, y_tr, y_val = train_test_split(X_auto, y_auto, test_size=0.5, random_state=0)

degrees = range(1, 11)
val_mse = []
for d in degrees:
    pipe = Pipeline([
        ('poly', PolynomialFeatures(d, include_bias=False)),
        ('lr',   LinearRegression()),
    ])
    pipe.fit(X_tr, y_tr)
    val_mse.append(mean_squared_error(y_val, pipe.predict(X_val)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: single split
axes[0].plot(degrees, val_mse, 'o-', color='tomato')
axes[0].set_xlabel('Polynomial degree')
axes[0].set_ylabel('Validation MSE')
axes[0].set_title('Single validation-set split')

# Right: 10 different random splits — notice the variability
for seed in range(10):
    X_tr2, X_val2, y_tr2, y_val2 = train_test_split(
        X_auto, y_auto, test_size=0.5, random_state=seed
    )
    curve = []
    for d in degrees:
        pipe = Pipeline([
            ('poly', PolynomialFeatures(d, include_bias=False)),
            ('lr',   LinearRegression()),
        ])
        pipe.fit(X_tr2, y_tr2)
        curve.append(mean_squared_error(y_val2, pipe.predict(X_val2)))
    axes[1].plot(degrees, curve, alpha=0.5)

axes[1].set_xlabel('Polynomial degree')
axes[1].set_ylabel('Validation MSE')
axes[1].set_title('10 different random splits — high variability')

for ax in axes:
    ax.set_xticks(range(1, 11))
plt.tight_layout()
plt.show()


**Key observations:**
- The single split correctly identifies degree 2 as a good choice (validation MSE drops from degree 1 to 2, then stays flat).
- Different random splits give wildly different MSE values and even different optimal degrees.
- This high variance is the main drawback of the validation-set approach.

**Why does it also overestimate test error?** Because we train on only half the data.
A model trained on all 392 observations will fit better than one trained on 196.
The validation error overestimates how badly the full-data model performs.

## 3. K-fold cross-validation

### The idea

Divide the data into K equal-sized **folds**. Leave out fold k, train on the remaining
K−1 folds, evaluate on fold k. Repeat for every fold. Average the K MSEs.

$$\text{CV}_{(K)} = \sum_{k=1}^{K} \frac{n_k}{n} \cdot \text{MSE}_k$$

Every observation is used for validation exactly once. More data goes into training,
so the estimates are less biased and less variable than a single split.

In [ ]:
# sklearn cross_val_score: the clean API
degrees = range(1, 11)
cv5_mse, cv10_mse = [], []

for d in degrees:
    pipe = Pipeline([
        ('poly', PolynomialFeatures(d, include_bias=False)),
        ('lr',   LinearRegression()),
    ])
    # cross_val_score returns negative MSE (sklearn convention: higher = better)
    cv5  = -cross_val_score(pipe, X_auto, y_auto, cv=5,  scoring='neg_mean_squared_error')
    cv10 = -cross_val_score(pipe, X_auto, y_auto, cv=10, scoring='neg_mean_squared_error')
    cv5_mse.append(cv5.mean())
    cv10_mse.append(cv10.mean())

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(degrees, cv5_mse,  'o-', label='5-fold CV',  color='steelblue')
ax.plot(degrees, cv10_mse, 's-', label='10-fold CV', color='orange')
ax.set_xlabel('Polynomial degree')
ax.set_ylabel('CV MSE')
ax.set_title('5-fold vs 10-fold CV on Auto data')
ax.set_xticks(range(1, 11))
ax.legend()
plt.tight_layout()
plt.show()

print(f'5-fold  best degree: {degrees[np.argmin(cv5_mse)]}')
print(f'10-fold best degree: {degrees[np.argmin(cv10_mse)]}')


### Manual K-fold to see the formula in action

In [ ]:
# Implement CV_(K) = sum(n_k/n * MSE_k) manually
def manual_cv(X, y, degree, K=10):
    kf = KFold(n_splits=K, shuffle=True, random_state=0)
    n = len(y)
    total = 0.0
    for train_idx, val_idx in kf.split(X):
        pipe = Pipeline([
            ('poly', PolynomialFeatures(degree, include_bias=False)),
            ('lr',   LinearRegression()),
        ])
        pipe.fit(X[train_idx], y[train_idx])
        mse_k = mean_squared_error(y[val_idx], pipe.predict(X[val_idx]))
        n_k = len(val_idx)
        total += (n_k / n) * mse_k
    return total

d = 2
manual_result = manual_cv(X_auto, y_auto, degree=d, K=10)

# Compare with sklearn cross_val_score
pipe2 = Pipeline([
    ('poly', PolynomialFeatures(d, include_bias=False)),
    ('lr',   LinearRegression()),
])
sklearn_result = -cross_val_score(
    pipe2, X_auto, y_auto, cv=KFold(10, shuffle=True, random_state=0),
    scoring='neg_mean_squared_error'
).mean()

print(f'Manual CV_(10) for degree={d}: {manual_result:.4f}')
print(f'sklearn CV_(10) for degree={d}: {sklearn_result:.4f}')
print('(Small differences due to how sklearn computes weighted vs unweighted average)')


### Leave-One-Out CV (LOOCV)

In [ ]:
# LOOCV via sklearn (cv=len(X) or LeaveOneOut)
# This takes a moment — 392 model fits
loocv_mse = []
for d in degrees:
    pipe = Pipeline([
        ('poly', PolynomialFeatures(d, include_bias=False)),
        ('lr',   LinearRegression()),
    ])
    scores = -cross_val_score(
        pipe, X_auto, y_auto,
        cv=LeaveOneOut(),
        scoring='neg_mean_squared_error',
    )
    loocv_mse.append(scores.mean())

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(degrees, loocv_mse, 'o-',  label='LOOCV',    color='steelblue')
ax.plot(degrees, cv10_mse,  's--', label='10-fold CV', color='orange')
ax.set_xlabel('Polynomial degree')
ax.set_ylabel('CV MSE')
ax.set_title('LOOCV vs 10-fold CV — both find minimum at degree 2')
ax.set_xticks(range(1, 11))
ax.legend()
plt.tight_layout()
plt.show()

print(f'LOOCV  best degree: {degrees[np.argmin(loocv_mse)]}')
print(f'10-fold best degree: {degrees[np.argmin(cv10_mse)]}')


### LOOCV shortcut for OLS — the leverage formula

For least-squares regression there is an amazing shortcut:

$$\text{CV}_{(n)} = \frac{1}{n} \sum_{i=1}^{n} \left( \frac{y_i - \hat{y}_i}{1 - h_i} \right)^2$$

where $h_i$ is the **leverage** — the $i$-th diagonal of the hat matrix
$H = X(X^TX)^{-1}X^T$.

This requires fitting the model **only once** on all the data.

In [ ]:
# Verify the LOOCV shortcut for degree=2
from numpy.linalg import lstsq

d = 2
poly = PolynomialFeatures(d, include_bias=True)
X_poly = poly.fit_transform(X_auto)         # design matrix, shape (n, d+1)

# Hat matrix diagonal
H = X_poly @ np.linalg.pinv(X_poly.T @ X_poly) @ X_poly.T
h = np.diag(H)  # leverage values

# Fitted values from full OLS
lr = LinearRegression(fit_intercept=False)  # intercept already in X_poly
lr.fit(X_poly, y_auto)
y_hat = lr.predict(X_poly)

# LOOCV via the shortcut
resid = y_auto - y_hat
loocv_shortcut = np.mean((resid / (1 - h))**2)

# LOOCV via actual fitting (computed above)
loocv_actual = loocv_mse[d - 1]

print(f'LOOCV (shortcut formula): {loocv_shortcut:.6f}')
print(f'LOOCV (actual K=n fits):  {loocv_actual:.6f}')
print(f'Match: {np.isclose(loocv_shortcut, loocv_actual, rtol=1e-3)}')


### CV for classification — SE of fold errors

In [ ]:
# Iris: 3-class LogisticRegression, 10-fold CV
from sklearn.datasets import load_iris

iris = load_iris()
X_iris, y_iris = iris.data, iris.target

pipe_iris = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=1000)),
])

K = 10
cv_scores = cross_val_score(
    pipe_iris, X_iris, y_iris,
    cv=KFold(n_splits=K, shuffle=True, random_state=42),
    scoring='accuracy',
)

# Each fold error = 1 - accuracy
fold_errors = 1 - cv_scores
cv_err = fold_errors.mean()

# SE formula from the PDF (approximate — fold errors are correlated)
se_est = np.sqrt((1/K) * np.sum((fold_errors - fold_errors.mean())**2) / (K - 1))

print(f'10-fold CV error: {cv_err:.4f} ± {se_est:.4f} (estimated SE)')
print(f'Fold errors: {fold_errors.round(4)}')
print()
print('Note: the SE formula treats fold errors as independent, but they share')
print('training data — so this SE is slightly too small (anti-conservative).')


### GridSearchCV — choosing hyperparameters with CV

In [ ]:
# Choose k in KNN using 10-fold CV on iris
ks = [1, 3, 5, 7, 11, 21, 51]

grid = GridSearchCV(
    Pipeline([
        ('scaler', StandardScaler()),
        ('knn',    KNeighborsClassifier()),
    ]),
    param_grid={'knn__n_neighbors': ks},
    cv=10,
    scoring='accuracy',
    refit=True,
)
grid.fit(X_iris, y_iris)

results = pd.DataFrame(grid.cv_results_)
print(results[['param_knn__n_neighbors', 'mean_test_score', 'std_test_score']].to_string(index=False))
print(f'\nBest k = {grid.best_params_["knn__n_neighbors"]}  '
      f'(CV accuracy = {grid.best_score_:.4f})')


## 4. CV: right and wrong

### The classic leakage mistake

Consider a two-step classifier:
1. Select the 100 features most correlated with the class labels (from all 5000).
2. Fit logistic regression on those 100 features.

**Wrong:** apply CV only to step 2 (reuse the same 100 features selected from ALL data).

Step 1 has already *seen* the labels for every observation. The selected features are
chosen to correlate with labels in this *specific sample* — even if all 5000 are random
noise. Applying CV only to the classifier gives near-zero error, even though the true
error is 50%.

In [ ]:
# The wrong way: feature selection outside the CV loop
n_samples, n_features, n_select = 50, 5000, 100
X_noise = rng.standard_normal((n_samples, n_features))
y_noise = rng.integers(0, 2, n_samples)  # completely random binary labels

# Step 1: select top-100 features by correlation with y (using ALL data!)
corr = np.abs(X_noise.T @ (y_noise - y_noise.mean()))
top100 = np.argsort(corr)[-n_select:]
X_selected = X_noise[:, top100]  # shape (50, 100)

# Step 2: CV on the pre-selected features
wrong_cv_error = 1 - cross_val_score(
    LogisticRegression(max_iter=1000, C=0.01),
    X_selected, y_noise, cv=5, scoring='accuracy'
).mean()

print(f'TRUE error (random labels): ~50%')
print(f'WRONG CV error:              {wrong_cv_error*100:.1f}%')
print()
print('The classifier appears excellent — but only because step 1 peeked at the labels.')


In [ ]:
# The right way: wrap EVERYTHING in a Pipeline
pipe_right = Pipeline([
    ('select', SelectKBest(f_classif, k=100)),   # selection happens inside each fold
    ('clf',    LogisticRegression(max_iter=1000, C=0.01)),
])

right_cv_error = 1 - cross_val_score(
    pipe_right, X_noise, y_noise, cv=5, scoring='accuracy'
).mean()

print(f'TRUE error (random labels):  ~50%')
print(f'WRONG CV error (step 2 only): {wrong_cv_error*100:.1f}%  ← leakage!')
print(f'RIGHT CV error (full pipeline): {right_cv_error*100:.1f}%  ← honest')


**Rule:** everything that uses the labels — scaling, feature selection, dimensionality
reduction — must live inside the `Pipeline` so it runs separately on each training fold.

This mistake has been made in many high-profile genomics papers (Ambroise & McLachlan 2002).
The 5000 noise features look predictive because their spurious correlations in the training
sample are exploited in step 1, then validated on the same sample in step 2.

## 5. The bootstrap

### The investment example

Invest fraction α in asset X and 1−α in asset Y. To minimise portfolio variance:

$$\alpha^* = \frac{\sigma_Y^2 - \sigma_{XY}}{\sigma_X^2 + \sigma_Y^2 - 2\sigma_{XY}}$$

With $\sigma_X^2 = 1$, $\sigma_Y^2 = 1.25$, $\sigma_{XY} = 0.5$, the true α = 0.6.

We don't know these population parameters — we estimate them from a sample.
How accurate is our estimate $\hat{\alpha}$?

In [ ]:
# True population parameters
sig2_X, sig2_Y, sig_XY = 1.0, 1.25, 0.5
alpha_true = (sig2_Y - sig_XY) / (sig2_X + sig2_Y - 2*sig_XY)
print(f'True alpha = {alpha_true:.4f}')

def alpha_hat(X, Y):
    """Estimate optimal allocation alpha from samples X and Y."""
    s2x = np.var(X, ddof=1)
    s2y = np.var(Y, ddof=1)
    sxy = np.cov(X, Y, ddof=1)[0, 1]
    return (s2y - sxy) / (s2x + s2y - 2*sxy)

# Generate one sample of n=100 from the bivariate normal
cov_matrix = [[sig2_X, sig_XY], [sig_XY, sig2_Y]]
n = 100
sample = rng.multivariate_normal([0, 0], cov_matrix, size=n)
X_one, Y_one = sample[:, 0], sample[:, 1]
print(f'alpha_hat from one sample (n=100): {alpha_hat(X_one, Y_one):.4f}')


In [ ]:
# Simulate from the true population: 1000 datasets, compute alpha_hat for each
n_sim = 1000
alphas_true_dist = np.array([
    alpha_hat(*rng.multivariate_normal([0, 0], cov_matrix, size=n).T)
    for _ in range(n_sim)
])

print(f'Mean of 1000 alpha_hat estimates: {alphas_true_dist.mean():.4f}  (true = {alpha_true:.4f})')
print(f'SD   of 1000 alpha_hat estimates: {alphas_true_dist.std():.4f}  ← true SE(alpha_hat)')


In [ ]:
# Bootstrap from ONE dataset: resample with replacement B times
B = 1000
alphas_boot = []
for _ in range(B):
    idx = rng.integers(0, n, size=n)  # sample indices with replacement
    alphas_boot.append(alpha_hat(X_one[idx], Y_one[idx]))
alphas_boot = np.array(alphas_boot)

se_boot = alphas_boot.std(ddof=1)
print(f'Bootstrap SE_B(alpha_hat): {se_boot:.4f}')
print(f'True      SE(alpha_hat):   {alphas_true_dist.std():.4f}')
print()

# Visualise: true distribution vs bootstrap distribution
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].hist(alphas_true_dist, bins=30, color='orange', edgecolor='white')
axes[0].axvline(alpha_true, color='red', linewidth=2, label=f'True α={alpha_true}')
axes[0].set_title('True distribution\n(1000 simulated datasets)')
axes[0].set_xlabel('α̂')
axes[0].legend(fontsize=9)

axes[1].hist(alphas_boot, bins=30, color='steelblue', edgecolor='white')
axes[1].axvline(alpha_true, color='red', linewidth=2, label=f'True α={alpha_true}')
axes[1].set_title(f'Bootstrap distribution\n({B} resamples of one dataset)')
axes[1].set_xlabel('α̂*')
axes[1].legend(fontsize=9)

axes[2].boxplot([alphas_true_dist, alphas_boot],
                labels=['True', 'Bootstrap'],
                patch_artist=True,
                boxprops=dict(facecolor='lightgrey'))
axes[2].axhline(alpha_true, color='red', linewidth=2, label=f'True α={alpha_true}')
axes[2].set_title('Comparison')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()


### Bootstrap percentile confidence interval

In [ ]:
# 90% CI: 5th and 95th percentiles of the bootstrap distribution
ci_90 = np.quantile(alphas_boot, [0.05, 0.95])
print(f'Bootstrap 90% CI for alpha: ({ci_90[0]:.3f}, {ci_90[1]:.3f})')
print(f'True alpha = {alpha_true:.3f}  — inside the interval: {ci_90[0] <= alpha_true <= ci_90[1]}')

# 95% CI
ci_95 = np.quantile(alphas_boot, [0.025, 0.975])
print(f'Bootstrap 95% CI for alpha: ({ci_95[0]:.3f}, {ci_95[1]:.3f})')


### Bootstrap for regression coefficients

Bootstrap works for **any** estimator. Here we compare bootstrap SE of OLS
coefficients to the standard OLS formula.

In [ ]:
# Diabetes dataset — OLS coefficient SE: bootstrap vs analytical
from sklearn.datasets import load_diabetes

diabetes = load_diabetes()
X_dia = diabetes.data     # 442 × 10, already standardised
y_dia = diabetes.target
n_dia = len(y_dia)

# Bootstrap SE
B = 1000
boot_coefs = []
for _ in range(B):
    idx = rng.integers(0, n_dia, size=n_dia)
    lr_b = LinearRegression().fit(X_dia[idx], y_dia[idx])
    boot_coefs.append(lr_b.coef_)
boot_coefs = np.array(boot_coefs)
se_boot_dia = boot_coefs.std(axis=0, ddof=1)

# Analytical OLS SE (from the hat matrix)
lr_full = LinearRegression().fit(X_dia, y_dia)
y_pred_dia = lr_full.predict(X_dia)
residuals  = y_dia - y_pred_dia
s2         = np.sum(residuals**2) / (n_dia - X_dia.shape[1] - 1)
Xd = np.column_stack([np.ones(n_dia), X_dia])
se_ols_dia = np.sqrt(s2 * np.diag(np.linalg.inv(Xd.T @ Xd)))[1:]  # skip intercept

print(f'{"Feature":<10}  {"Bootstrap SE":>12}  {"OLS SE":>10}')
for i, name in enumerate(diabetes.feature_names):
    print(f'{name:<10}  {se_boot_dia[i]:>12.4f}  {se_ols_dia[i]:>10.4f}')


Bootstrap SE and OLS analytical SE are very close. The bootstrap is particularly
valuable when no analytical formula exists (tree ensembles, custom statistics, etc.).

### scipy.stats.bootstrap — the modern API

In [ ]:
# scipy.stats.bootstrap: 95% CI for the median of flipper_length_mm
penguins = load_dataset('penguins').dropna(subset=['flipper_length_mm'])
flipper  = penguins['flipper_length_mm'].to_numpy()

result = scipy.stats.bootstrap(
    (flipper,),               # data as a tuple of arrays
    statistic=np.median,
    n_resamples=2000,
    confidence_level=0.95,
    method='percentile',      # 'BCa' for bias-corrected accelerated
    random_state=0,
)

ci = result.confidence_interval
print(f'Sample median: {np.median(flipper):.1f} mm')
print(f'Bootstrap 95% CI (percentile): ({ci.low:.1f}, {ci.high:.1f}) mm')
print(f'Bootstrap SE: {result.standard_error:.3f} mm')

# Normal-approximation CI for comparison
se_normal = scipy.stats.sem(flipper)  # this is SE of the mean, not median
mean_flipper = np.mean(flipper)
ci_normal = (mean_flipper - 1.96*se_normal, mean_flipper + 1.96*se_normal)
print(f'\nFor reference — normal CI for the MEAN: ({ci_normal[0]:.1f}, {ci_normal[1]:.1f}) mm')
print('(Bootstrap CI is for the MEDIAN, which has no simple SE formula)')


### Why bootstrap is bad for prediction error

In [ ]:
# Demonstrate: bootstrap severely underestimates prediction error
# because ~63% of training obs appear in each bootstrap sample

# Fraction of observations NOT in a bootstrap sample of size n
# = (1 - 1/n)^n → 1/e ≈ 36.8% as n→∞
ns = [10, 50, 100, 392, 1000, 10000]
fracs = [(1 - 1/n)**n for n in ns]
print('n       | Fraction out-of-bag')
for n, f in zip(ns, fracs):
    print(f'{n:7d} | {f:.4f}')
print(f'\nLimit (1/e): {1/np.e:.4f}')
print('→ ~63.2% of obs appear in each bootstrap sample')
print('→ Validation on the full dataset after training on bootstrap samples')
print('  is not a fair test — the model has seen most of the test set.')

# Numerical comparison on Auto data
B = 100
boot_pred_errors = []
for _ in range(B):
    idx = rng.integers(0, len(X_auto), size=len(X_auto))
    pipe = Pipeline([
        ('poly', PolynomialFeatures(2, include_bias=False)),
        ('lr',   LinearRegression()),
    ])
    pipe.fit(X_auto[idx], y_auto[idx])
    # Test on the FULL original dataset (contains ~63% of the training obs)
    boot_pred_errors.append(mean_squared_error(y_auto, pipe.predict(X_auto)))

cv10_degree2 = cv10_mse[1]  # 10-fold CV MSE for degree=2 (computed earlier)

print(f'\nAuto data, degree=2 polynomial:')
print(f'Bootstrap prediction error (train on boot, test on all): {np.mean(boot_pred_errors):.3f}')
print(f'10-fold CV prediction error (honest estimate):            {cv10_degree2:.3f}')
print('Bootstrap estimate is smaller — it underestimates error due to overlap.')


### Recap
- **Training error systematically underestimates test error** — never use it to select models.
- **Validation set** is simple but has high variance; overestimates error because trains on less data.
- **K-fold CV** (K = 5 or 10) is the standard: uses data efficiently, lower variance than a single split.
- **LOOCV** is nearly unbiased but has high variance and is slow (K refits).
- **CV must wrap all preprocessing steps** — use sklearn `Pipeline` to prevent data leakage.
- **Bootstrap** quantifies SE and CI for any estimator via sampling with replacement; use `scipy.stats.bootstrap`.
- **Bootstrap ≠ prediction error** — use CV for that; bootstrap for uncertainty quantification.

---

# Exercises

Each exercise is followed by its solution.
Try to solve the tasks yourself before reading the solution cell.

---

## Exercise 1

**Task 1.** Load `penguins`, drop NaN. Using only `bill_length_mm` and `bill_depth_mm`,
fit a `LogisticRegression` to predict `species`.

1. Use a single 70/30 stratified train/test split and report test accuracy.
2. Use 10-fold CV and report mean CV accuracy ± estimated SE.
3. Compare the two estimates — how much do they differ?

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, KFold
import numpy as np
# your code here


### Exercise 1 — Solution

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, KFold
import numpy as np

df = load_dataset('penguins').dropna(subset=['bill_length_mm', 'bill_depth_mm', 'species'])
X = df[['bill_length_mm', 'bill_depth_mm']].to_numpy()
y = df['species'].to_numpy()

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=1000)),
])

# 1. Single 70/30 split
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
pipe.fit(Xtr, ytr)
split_acc = pipe.score(Xte, yte)
print(f'Single 70/30 split accuracy: {split_acc:.4f}')

# 2. 10-fold CV
K = 10
cv_scores = cross_val_score(
    pipe, X, y,
    cv=KFold(n_splits=K, shuffle=True, random_state=42),
    scoring='accuracy',
)
fold_errors = 1 - cv_scores
cv_acc  = cv_scores.mean()
cv_se   = np.sqrt((1/K) * np.sum((fold_errors - fold_errors.mean())**2) / (K-1))
print(f'10-fold CV accuracy:         {cv_acc:.4f} ± {cv_se:.4f}')
print(f'Difference (CV - split):     {cv_acc - split_acc:+.4f}')


---

## Exercise 2

**Task 2.** Using the Auto dataset (`mpg` ~ `horsepower`), plot CV MSE vs polynomial
degree for **K ∈ {2, 5, 10, LOOCV}** on the same axes.

- Which K gives the most stable minimum?
- Does LOOCV always agree with 10-fold CV about the best degree?

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold, LeaveOneOut
from sklearn.metrics import mean_squared_error
# your code here


### Exercise 2 — Solution

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold, LeaveOneOut

auto = sns.load_dataset('mpg').dropna(subset=['mpg', 'horsepower'])
X_a = auto['horsepower'].to_numpy().reshape(-1, 1)
y_a = auto['mpg'].to_numpy()
degrees = range(1, 11)

configs = [
    ('K=2',    KFold(2,  shuffle=True, random_state=0), 'C0'),
    ('K=5',    KFold(5,  shuffle=True, random_state=0), 'C1'),
    ('K=10',   KFold(10, shuffle=True, random_state=0), 'C2'),
    ('LOOCV',  LeaveOneOut(),                           'C3'),
]

fig, ax = plt.subplots(figsize=(8, 5))
for label, cv, color in configs:
    mse_list = []
    for d in degrees:
        pipe = Pipeline([
            ('poly', PolynomialFeatures(d, include_bias=False)),
            ('lr',   LinearRegression()),
        ])
        scores = -cross_val_score(pipe, X_a, y_a, cv=cv,
                                   scoring='neg_mean_squared_error')
        mse_list.append(scores.mean())
    best = degrees[np.argmin(mse_list)]
    ax.plot(degrees, mse_list, 'o-', color=color, label=f'{label} (best={best})')

ax.set_xlabel('Polynomial degree')
ax.set_ylabel('CV MSE')
ax.set_title('CV MSE vs Polynomial Degree for K ∈ {2, 5, 10, LOOCV}')
ax.set_xticks(range(1, 11))
ax.legend()
plt.tight_layout()
plt.show()


---

## Exercise 3

**Task 3.** Reproduce the data-leakage demo:

- Generate 5000 noise features, 50 samples, binary random labels.
- **Wrong way:** select top-20 features by correlation, then run 5-fold CV on `LogisticRegression`.
- **Right way:** use `Pipeline(SelectKBest(k=20) + LogisticRegression)` with 5-fold CV.
- Print both CV errors and the true error (~50%). Run 10 times and average to reduce noise.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
import numpy as np
# your code here


### Exercise 3 — Solution

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
import numpy as np

rng3 = np.random.default_rng(7)
n_runs = 10
wrong_errors, right_errors = [], []

for _ in range(n_runs):
    X_n = rng3.standard_normal((50, 5000))
    y_n = rng3.integers(0, 2, 50)

    # Wrong: select outside CV loop
    corr = np.abs(X_n.T @ (y_n - y_n.mean()))
    top20_idx = np.argsort(corr)[-20:]
    wrong_cv = 1 - cross_val_score(
        LogisticRegression(C=0.01, max_iter=1000),
        X_n[:, top20_idx], y_n, cv=5, scoring='accuracy'
    ).mean()
    wrong_errors.append(wrong_cv)

    # Right: Pipeline wraps selection inside each fold
    pipe = Pipeline([
        ('sel', SelectKBest(f_classif, k=20)),
        ('clf', LogisticRegression(C=0.01, max_iter=1000)),
    ])
    right_cv = 1 - cross_val_score(pipe, X_n, y_n, cv=5, scoring='accuracy').mean()
    right_errors.append(right_cv)

print(f'True error (random labels):           ~50%')
print(f'WRONG CV error (avg over {n_runs} runs): {np.mean(wrong_errors)*100:.1f}%  ← leakage!')
print(f'RIGHT CV error (avg over {n_runs} runs): {np.mean(right_errors)*100:.1f}%  ← honest')


---

## Exercise 4

**Task 4.** Use `scipy.stats.bootstrap` to compute a 95% CI for the **median**
of `flipper_length_mm` in the penguins dataset.

1. Use `method='percentile'` with `n_resamples=2000`.
2. Also try `method='BCa'` (bias-corrected accelerated). Does the interval change?
3. For comparison, compute a normal-approximation CI for the **mean**.
   Why can we not simply use the normal approximation for the median?

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import numpy as np
import scipy.stats
# your code here


### Exercise 4 — Solution

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import numpy as np
import scipy.stats

df4 = load_dataset('penguins').dropna(subset=['flipper_length_mm'])
flipper4 = df4['flipper_length_mm'].to_numpy()
print(f'n = {len(flipper4)},  sample median = {np.median(flipper4):.1f} mm')

# 1. Percentile bootstrap CI
res_pct = scipy.stats.bootstrap(
    (flipper4,), statistic=np.median, n_resamples=2000,
    confidence_level=0.95, method='percentile', random_state=0
)
print(f'\n95% CI (percentile): ({res_pct.confidence_interval.low:.1f}, {res_pct.confidence_interval.high:.1f}) mm')

# 2. BCa bootstrap CI
res_bca = scipy.stats.bootstrap(
    (flipper4,), statistic=np.median, n_resamples=2000,
    confidence_level=0.95, method='BCa', random_state=0
)
print(f'95% CI (BCa):        ({res_bca.confidence_interval.low:.1f}, {res_bca.confidence_interval.high:.1f}) mm')

# 3. Normal approximation CI for the MEAN (not median)
mean4  = flipper4.mean()
se4    = scipy.stats.sem(flipper4)
ci_norm = scipy.stats.t.interval(0.95, df=len(flipper4)-1, loc=mean4, scale=se4)
print(f'\n95% CI (normal, for MEAN): ({ci_norm[0]:.1f}, {ci_norm[1]:.1f}) mm')
print(f'Sample mean = {mean4:.1f} mm')
print()
print('Why not normal approx for the median?')
print(' - The CLT gives SE(mean) = σ/√n analytically.')
print(' - SE(median) requires knowing the population density at the median — usually unknown.')
print(' - Bootstrap handles this without any distributional assumption.')
